# SELLERS - INCREMENTAL LOAD WITH AUTO LOADER

## CREATED BY: RAHUL M
## CREATED DATE: 20260822
## DESCRIPTION: Incremental ingestion from landing using Auto Loader

In [0]:
%run /Workspace/Users/rms181800@gmail.com/AZURE_B3_PROJECT_AUG/FUNCTIONS/functions

In [0]:
from pyspark.sql.functions import current_timestamp

df_sellers_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/retail_project_b3/bronze/_checkpoints/sellers_schema") \
    .load("/Volumes/retail_project_b3/landing/raw_data/olist_sellers_dataset.csv") \
    .withColumn("ingest_ts", current_timestamp())

print("✅ Auto Loader initialized for sellers")

In [0]:
query = df_sellers_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/retail_project_b3/bronze/_checkpoints/sellers") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .start("/Volumes/retail_project_b3/bronze/sellers/")

query.awaitTermination()
if query.lastProgress:
    print(f"✅ Processed {query.lastProgress.get('numInputRows', 0)} records")
else:
    print("✅ No new files to process")